# 低評価予測モデルの比較と最終選択

## 目的

これまでに作成した特徴量を用いて、低評価予測モデルの最終的な特徴量構成を検討する。

ロジスティック回帰を基準モデルとし、その後Random ForestやBoosting系モデルと比較する。

## 評価方針

- Train：2017年12月まで
- Validation：2018年1月〜4月
- Holdout：2018年5月〜8月
- 特徴量選択・しきい値決定にはValidationを使用する
- Holdoutは最終モデル決定後に一度だけ評価する

## 評価指標

- F1
- Precision
- Recall
- ROC-AUC
- PR-AUC（Average Precision）

低評価レビュー（1〜2点）を陽性クラスとする。

In [14]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Notebookの作業ディレクトリを確認
current_dir = Path.cwd().resolve()

# プロジェクトルートを取得
if (current_dir / "src" / "project_paths.py").is_file():
    project_root = current_dir
elif (current_dir.parent / "src" / "project_paths.py").is_file():
    project_root = current_dir.parent
else:
    raise FileNotFoundError(
        "プロジェクトルートが見つかりません。"
        "Notebookの実行ディレクトリを確認してください。"
    )

# モジュール検索パスに追加
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 既存の共通パス設定を使用
from src.project_paths import DATA_DIR, PROJECT_ROOT

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)


PROJECT_ROOT: /workspaces/AWS_Python_data_analysis
DATA_DIR: /workspaces/AWS_Python_data_analysis/data


In [15]:
df_model = pd.read_csv(DATA_DIR / "model_improved_6.csv")

print("Shape:", df_model.shape)

print("\n=== 列一覧 ===")
print(df_model.columns.tolist())

print("\n=== order_idの重複数 ===")
print(df_model["order_id"].duplicated().sum())

print("\n=== 欠損数 ===")
print(df_model.isna().sum())

print("\n=== 購入月の範囲 ===")
print(df_model["purchase_month"].min())
print(df_model["purchase_month"].max())


Shape: (95824, 29)

=== 列一覧 ===
['order_id', 'review_score', 'total_price', 'total_freight', 'delivery_delay_days', 'purchase_month', 'customer_state', 'freight_ratio', 'purchase_month_num', 'item_count', 'category_count', 'low_review', 'payment_count', 'payment_type_count', 'main_payment_type', 'max_payment_installments', 'log_freight_ratio', 'month_sin', 'month_cos', 'main_category', 'main_category_gmv', 'customer_unique_id', 'prior_order_count', 'is_repeat_customer', 'prior_customer_gmv', 'prior_avg_order_value', 'days_since_previous_order', 'log_prior_customer_gmv', 'log_prior_avg_order_value']

=== order_idの重複数 ===
0

=== 欠損数 ===
order_id                     0
review_score                 0
total_price                  0
total_freight                0
delivery_delay_days          0
purchase_month               0
customer_state               0
freight_ratio                0
purchase_month_num           0
item_count                   0
category_count               0
low_review      

In [16]:
# Model A：注文・地域・購入時期
numeric_A = [
    "total_price",
    "total_freight",
    "delivery_delay_days",
    "log_freight_ratio",
    "item_count",
    "category_count",
    "month_sin",
    "month_cos",
]

categorical_A = [
    "customer_state",
]

# Model B：支払い特徴量を追加
numeric_B = numeric_A + [
    "payment_count",
    "payment_type_count",
    "max_payment_installments",
]

categorical_B = categorical_A + [
    "main_payment_type",
]

# Model C：商品カテゴリを追加
numeric_C = numeric_B.copy()

categorical_C = categorical_B + [
    "main_category",
]

# Model D：顧客履歴を追加
numeric_D = numeric_C + [
    "prior_order_count",
    "is_repeat_customer",
    "log_prior_customer_gmv",
    "log_prior_avg_order_value",
    "days_since_previous_order",
]

categorical_D = categorical_C.copy()

feature_sets = {
    "A_Order": (numeric_A, categorical_A),
    "B_Payment": (numeric_B, categorical_B),
    "C_Category": (numeric_C, categorical_C),
    "D_History": (numeric_D, categorical_D),
}

for name, (numeric, categorical) in feature_sets.items():
    print(
        name,
        "数値:",
        len(numeric),
        "カテゴリ:",
        len(categorical),
    )


A_Order 数値: 8 カテゴリ: 1
B_Payment 数値: 11 カテゴリ: 2
C_Category 数値: 11 カテゴリ: 3
D_History 数値: 16 カテゴリ: 3


In [17]:
# 時系列でTrain/Validationを分割

train = df_model[df_model["purchase_month"] <= "2017-12"].copy()

validation = df_model[
    (df_model["purchase_month"] >= "2018-01")
    & (df_model["purchase_month"] <= "2018-04")
].copy()

# Holdoutは分割件数のみ確認し、モデル比較では使用しない
holdout_count = (
    (df_model["purchase_month"] >= "2018-05")
    & (df_model["purchase_month"] <= "2018-08")
).sum()

print("Train:", len(train))
print("Validation:", len(validation))
print("Holdout（未使用）:", holdout_count)


Train: 43362
Validation: 27220
Holdout（未使用）: 25242


In [18]:
# モデルA-Dを同じ条件で評価する

from sklearn.metrics import average_precision_score, roc_auc_score

from src import model_metrics, model_training

results = []

for model_name, (numeric, categorical) in feature_sets.items():
    model, y_val, val_prob = model_training.train_logistic_model(
        train,
        validation,
        numeric,
        categorical,
    )

    threshold_results = model_metrics.threshold_table(
        y_val,
        val_prob,
        thresholds=np.arange(0.05, 0.51, 0.01),
    )

    best = threshold_results.sort_values(
        ["f1", "threshold"],
        ascending=[False, True],
    ).iloc[0]

    results.append(
        {
            "model": model_name,
            "threshold": best["threshold"],
            "precision": best["precision"],
            "recall": best["recall"],
            "f1": best["f1"],
            "roc_auc": roc_auc_score(y_val, val_prob),
            "pr_auc": average_precision_score(y_val, val_prob),
        }
    )

comparison = pd.DataFrame(results)

print(
    comparison.sort_values(
        "f1",
        ascending=False,
    ).to_string(index=False)
)

     model  threshold  precision   recall       f1  roc_auc   pr_auc
   A_Order       0.26   0.611806 0.502795 0.551970 0.766418 0.520047
 B_Payment       0.26   0.611656 0.502124 0.551504 0.767047 0.519549
C_Category       0.25   0.566923 0.510396 0.537176 0.770396 0.513088
 D_History       0.25   0.567756 0.508607 0.536557 0.770000 0.512457


## 時系列クロスバリデーション

学習期間を段階的に拡大するExpanding Window方式を採用し、
モデル評価の時期依存性と汎化性能を検証する。

2018年4月までのデータをfold1～fold4まで分解し、それぞれでTrain,Validationを作成する。2018年5〜8月のHoldoutは
このCVには使用しない。

この検討をする際に、レビューが確定していない注文が含まれている場合、その注文ラベルはその時点で利用できなかった可能性も考慮する必要がある。

In [19]:
# 時系列クロスバリデーションの期間設定
cv_periods = [
    {
        "fold": 1,
        "train_end": "2017-03",
        "val_start": "2017-04",
        "val_end": "2017-06",
    },
    {
        "fold": 2,
        "train_end": "2017-06",
        "val_start": "2017-07",
        "val_end": "2017-09",
    },
    {
        "fold": 3,
        "train_end": "2017-09",
        "val_start": "2017-10",
        "val_end": "2017-12",
    },
    {
        "fold": 4,
        "train_end": "2017-12",
        "val_start": "2018-01",
        "val_end": "2018-04",
    },
]

cv_splits = []
cv_summary = []

for period in cv_periods:
    train_idx = np.flatnonzero(
        train["purchase_month"].le(period["train_end"]).to_numpy()
    )

    val_idx = np.flatnonzero(
        train["purchase_month"]
        .between(period["val_start"], period["val_end"])
        .to_numpy()
    )

    # 学習・検証データが存在することを確認
    assert len(train_idx) > 0
    assert len(val_idx) > 0

    # 同じ注文が学習と検証に含まれていないことを確認
    assert len(set(train_idx) & set(val_idx)) == 0

    # 学習期間が検証期間より過去にあることを確認
    assert period["train_end"] < period["val_start"]

    cv_splits.append((train_idx, val_idx))

    cv_summary.append(
        {
            "fold": period["fold"],
            "train_end": period["train_end"],
            "val_start": period["val_start"],
            "val_end": period["val_end"],
            "train_count": len(train_idx),
            "val_count": len(val_idx),
            "train_low_review_rate": train.iloc[train_idx]["low_review"].mean(),
            "val_low_review_rate": train.iloc[val_idx]["low_review"].mean(),
        }
    )

df_cv_summary = pd.DataFrame(cv_summary)

display(df_cv_summary)


AssertionError: 

In [ ]:
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from src import model_training

cv_results = []

for fold, (train_idx, val_idx) in enumerate(cv_splits, start=1):
    fold_train = train.iloc[train_idx].copy()
    fold_val = train.iloc[val_idx].copy()

    model, y_val, val_prob = model_training.train_logistic_model(
        fold_train,
        fold_val,
        numeric_A,
        categorical_A,
    )

    # 前回のValidationで選択したModel Aのしきい値
    threshold = 0.26
    val_pred = (val_prob >= threshold).astype(int)

    cv_results.append(
        {
            "fold": fold,
            "train_count": len(fold_train),
            "val_count": len(fold_val),
            "precision": precision_score(y_val, val_pred, zero_division=0),
            "recall": recall_score(y_val, val_pred, zero_division=0),
            "f1": f1_score(y_val, val_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_val, val_prob),
            "ap": average_precision_score(y_val, val_prob),
        }
    )

df_cv_results = pd.DataFrame(cv_results)

display(df_cv_results)

print("\n=== CV平均 ===")
print(df_cv_results[["precision", "recall", "f1", "roc_auc", "ap"]].mean().round(4))

print("\n=== CV標準偏差 ===")
print(df_cv_results[["precision", "recall", "f1", "roc_auc", "ap"]].std().round(4))


In [ ]:
from src import model_metrics

threshold_diagnostics = []

for fold, (train_idx, val_idx) in enumerate(cv_splits, start=1):
    fold_train = train.iloc[train_idx].copy()
    fold_val = train.iloc[val_idx].copy()

    model, y_val, val_prob = model_training.train_logistic_model(
        fold_train,
        fold_val,
        numeric_A,
        categorical_A,
    )

    threshold_results = model_metrics.threshold_table(
        y_val,
        val_prob,
        thresholds=np.arange(0.05, 0.51, 0.01),
    )

    best = threshold_results.sort_values(
        ["f1", "threshold"],
        ascending=[False, True],
    ).iloc[0]

    threshold_diagnostics.append(
        {
            "fold": fold,
            "best_threshold_diagnostic": best["threshold"],
            "best_f1_diagnostic": best["f1"],
            "f1_at_026": df_cv_results.loc[df_cv_results["fold"] == fold, "f1"].iloc[0],
        }
    )

df_threshold_diagnostics = pd.DataFrame(threshold_diagnostics)

display(df_threshold_diagnostics)


In [ ]:
# 購入日時とレビュー投稿日時が異なることを踏まえた調整
from src.project_paths import DATA_DIR

df_review_anchor = pd.read_csv(DATA_DIR / "review_anchor.csv")

print("=== Shape ===")
print(df_review_anchor.shape)

print("\n=== 列一覧 ===")
print(df_review_anchor.columns.tolist())

print("\n=== 先頭5行 ===")
display(df_review_anchor.head())

print("\n=== order_id重複数 ===")
print(df_review_anchor["order_id"].duplicated().sum())

print("\n=== 欠損数 ===")
print(df_review_anchor.isna().sum())

print("\n=== モデルデータとのorder_id一致数 ===")
print(df_model["order_id"].isin(df_review_anchor["order_id"]).sum())

In [ ]:
# レビュー日時データとの一致状況を確認
df_anchor_check = df_model[
    ["order_id", "review_score", "low_review", "purchase_month"]
].merge(
    df_review_anchor[
        ["order_id", "review_score", "low_review", "review_timestamp"]
    ].rename(
        columns={
            "review_score": "anchor_review_score",
            "low_review": "anchor_low_review",
        }
    ),
    on="order_id",
    how="left",
    validate="one_to_one",
    indicator=True,
)

# 一致状況
print("=== 一致状況 ===")
print(df_anchor_check["_merge"].value_counts())

# 一致・不一致別の低評価率
print("\n=== 低評価率 ===")
display(
    df_anchor_check.groupby("_merge", observed=True).agg(
        order_count=("order_id", "size"),
        low_review_rate=("low_review", "mean"),
    )
)

# 一致した注文でラベルが一致するか
matched = df_anchor_check[df_anchor_check["_merge"] == "both"].copy()

print("\n=== レビュースコア不一致件数 ===")
print((matched["review_score"] != matched["anchor_review_score"]).sum())

print("\n=== 低評価ラベル不一致件数 ===")
print((matched["low_review"] != matched["anchor_low_review"]).sum())

# レビュー日時が購入月より前になっていないか
matched["review_timestamp"] = pd.to_datetime(
    matched["review_timestamp"],
    errors="coerce",
)

matched["purchase_month_start"] = pd.to_datetime(matched["purchase_month"] + "-01")

print("\n=== レビュー日時の変換失敗件数 ===")
print(matched["review_timestamp"].isna().sum())

print("\n=== レビュー日時が購入月初日より前の件数 ===")
print((matched["review_timestamp"] < matched["purchase_month_start"]).sum())


In [ ]:
from pathlib import Path

import pandas as pd

from src.project_paths import DATA_DIR

# 既存CSVの中から、必要な日時列を探す
date_keywords = [
    "purchase",
    "delivered",
    "delivery",
    "review",
    "timestamp",
    "date",
]

for path in sorted(DATA_DIR.glob("*.csv")):
    columns = pd.read_csv(path, nrows=0).columns.tolist()

    date_columns = [
        col
        for col in columns
        if any(keyword in col.lower() for keyword in date_keywords)
    ]

    if date_columns:
        print(f"\n=== {path.name} ===")
        print(date_columns)


In [ ]:
from src.project_paths import DATA_DIR

df_order_dates = pd.read_csv(
    DATA_DIR / "order_delivery_timestamps.csv",
    parse_dates=["purchase_timestamp", "delivered_timestamp"],
)

df_delivery_check = df_model[["order_id", "purchase_month", "low_review"]].merge(
    df_order_dates,
    on="order_id",
    how="left",
    validate="one_to_one",
)

# 配送完了月
df_delivery_check["delivered_month"] = (
    df_delivery_check["delivered_timestamp"].dt.to_period("M").astype("string")
)

print("=== モデルデータとの結合 ===")
print("行数:", len(df_delivery_check))
print("購入日時の欠損:", df_delivery_check["purchase_timestamp"].isna().sum())
print("配送完了日時の欠損:", df_delivery_check["delivered_timestamp"].isna().sum())

print("\n=== 購入月と配送完了月の比較 ===")
has_delivery = df_delivery_check["delivered_timestamp"].notna()

print(
    "購入月と配送完了月が異なる注文数:",
    (
        df_delivery_check.loc[has_delivery, "purchase_month"]
        != df_delivery_check.loc[has_delivery, "delivered_month"]
    ).sum(),
)

print("\n=== 配送完了日時の有無による低評価率 ===")
display(
    df_delivery_check.assign(has_delivery=has_delivery)
    .groupby("has_delivery")
    .agg(
        order_count=("order_id", "size"),
        low_review_rate=("low_review", "mean"),
    )
)

print("\n=== 配送完了日時が欠損している注文のステータス ===")
display(df_delivery_check.loc[~has_delivery, "order_status"].value_counts(dropna=False))


購入月と配送完了月が異なる件数が約37.7%存在するので、「配送完了後、低評価を予測するモデル」を作成するにあたって、これまでの購入月に基づくCVと、配送完了月に基づくCVでは、学習・検証対象が大きく変わる可能性がある。

また、配送完了日時より前に、レビューが回答されている可能性がないかどうかを確かめる必要がある。

In [20]:
# 配送完了日時を基準にした月別件数
df_delivery_month_summary = (
    df_delivery_check.groupby("delivered_month", dropna=False)
    .agg(
        order_count=("order_id", "size"),
        low_review_rate=("low_review", "mean"),
    )
    .reset_index()
    .sort_values("delivered_month")
)

print("=== 配送完了月別の注文件数と低評価率 ===")
display(df_delivery_month_summary)

print("\n=== 配送完了月の範囲 ===")
print("最小:", df_delivery_check["delivered_timestamp"].min())
print("最大:", df_delivery_check["delivered_timestamp"].max())

print("\n=== 2018年9月以降に配送完了した注文 ===")
after_august = df_delivery_check["delivered_timestamp"] >= pd.Timestamp("2018-09-01")

print("件数:", after_august.sum())
print("低評価率:", df_delivery_check.loc[after_august, "low_review"].mean())

print("\n=== 購入から配送完了までの日数 ===")
delivery_days = (
    df_delivery_check["delivered_timestamp"] - df_delivery_check["purchase_timestamp"]
).dt.total_seconds() / 86400

display(delivery_days.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))


NameError: name 'df_delivery_check' is not defined

In [ ]:
# レビュー日時の確認
df_review_timing = df_delivery_check[
    ["order_id", "purchase_timestamp", "delivered_timestamp", "low_review"]
].merge(
    df_review_anchor[["order_id", "review_timestamp"]],
    on="order_id",
    how="left",
    validate="one_to_one",
)

df_review_timing["review_timestamp"] = pd.to_datetime(
    df_review_timing["review_timestamp"],
    errors="coerce",
)

has_review_time = df_review_timing["review_timestamp"].notna()

# レビュー回答日時と配送完了日時の比較
review_before_delivery = has_review_time & (
    df_review_timing["review_timestamp"] < df_review_timing["delivered_timestamp"]
)

review_after_delivery = has_review_time & (
    df_review_timing["review_timestamp"] >= df_review_timing["delivered_timestamp"]
)

print("=== レビュー日時の確認 ===")
print("モデルデータ:", len(df_review_timing))
print("レビュー日時あり:", has_review_time.sum())
print("レビュー日時なし:", (~has_review_time).sum())

print("\n=== レビューと配送完了の前後関係 ===")
print("配送完了前にレビュー回答:", review_before_delivery.sum())
print("配送完了後にレビュー回答:", review_after_delivery.sum())

print("\n=== 配送完了前にレビューされた注文の低評価率 ===")
print(df_review_timing.loc[review_before_delivery, "low_review"].mean())

print("\n=== 配送完了からレビュー回答までの日数 ===")
days_to_review = (
    df_review_timing.loc[review_after_delivery, "review_timestamp"]
    - df_review_timing.loc[review_after_delivery, "delivered_timestamp"]
).dt.total_seconds() / 86400

display(days_to_review.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))


In [21]:
import pandas as pd

from src.project_paths import DATA_DIR

# STEP 50-3：レビュー日時の分類データを読み込む
df_review_timing_features = pd.read_csv(
    DATA_DIR / "review_timing_features.csv",
    parse_dates=[
        "delivered_timestamp",
        "first_review_timestamp",
        "last_review_timestamp",
    ],
)

print("=== レビュー日時データ ===")
print("行数:", len(df_review_timing_features))
print(
    "order_id重複:",
    df_review_timing_features["order_id"].duplicated().sum(),
)

# モデルデータと結合
df_timing_model = df_model[["order_id", "low_review", "review_score"]].merge(
    df_review_timing_features,
    on="order_id",
    how="left",
    validate="one_to_one",
    indicator=True,
)

print("\n=== モデルデータとの結合 ===")
print("結合後の行数:", len(df_timing_model))
print(df_timing_model["_merge"].value_counts())

print("\n=== レビュー日時分類別の注文件数・低評価率 ===")
timing_summary = (
    df_timing_model.groupby("timing_category", dropna=False)
    .agg(
        order_count=("order_id", "size"),
        low_review_count=("low_review", "sum"),
        low_review_rate=("low_review", "mean"),
        multiple_review_orders=(
            "review_count",
            lambda x: x.gt(1).sum(),
        ),
    )
    .reset_index()
    .sort_values("order_count", ascending=False)
)

display(timing_summary)

print("\n=== 分類別の全体比率 ===")
timing_summary["share"] = timing_summary["order_count"] / len(df_timing_model)

display(timing_summary[["timing_category", "order_count", "share"]])

print("\n=== レビュー日時の欠損 ===")
print(
    "最初のレビュー日時なし:",
    df_timing_model["first_review_timestamp"].isna().sum(),
)
print(
    "最後のレビュー日時なし:",
    df_timing_model["last_review_timestamp"].isna().sum(),
)


=== レビュー日時データ ===
行数: 98673
order_id重複: 0

=== モデルデータとの結合 ===
結合後の行数: 95824
_merge
both          95824
left_only         0
right_only        0
Name: count, dtype: int64

=== レビュー日時分類別の注文件数・低評価率 ===


,timing_category,order_count,low_review_count,low_review_rate,multiple_review_orders
0,all_reviews_after_delivery,87683,7758,0.088478,355
1,all_reviews_before_delivery,4390,3417,0.778360,23
2,review_timestamp_missing,3646,1055,0.289358,42
3,reviews_before_and_after_delivery,105,6,0.057143,105



=== 分類別の全体比率 ===


,timing_category,order_count,share
0,all_reviews_after_delivery,87683,0.915042
1,all_reviews_before_delivery,4390,0.045813
2,review_timestamp_missing,3646,0.038049
3,reviews_before_and_after_delivery,105,0.001096



=== レビュー日時の欠損 ===
最初のレビュー日時なし: 3613
最後のレビュー日時なし: 3613


In [22]:
print("=== レビュー件数の分布 ===")

review_count_summary = (
    df_timing_model.groupby("review_count", dropna=False)
    .agg(
        order_count=("order_id", "size"),
        low_review_rate=("low_review", "mean"),
    )
    .reset_index()
    .sort_values("review_count")
)

display(review_count_summary)

print("\n=== 複数レビュー注文の分類 ===")

multiple_review_summary = (
    df_timing_model.loc[df_timing_model["review_count"] > 1]
    .groupby("timing_category", dropna=False)
    .agg(
        order_count=("order_id", "size"),
        low_review_rate=("low_review", "mean"),
    )
    .reset_index()
)

display(multiple_review_summary)


=== レビュー件数の分布 ===


,review_count,order_count,low_review_rate
0,1,95299,0.127903
1,2,521,0.088292
2,3,4,0.250000



=== 複数レビュー注文の分類 ===


,timing_category,order_count,low_review_rate
0,all_reviews_after_delivery,355,0.059155
1,all_reviews_before_delivery,23,0.565217
2,review_timestamp_missing,42,0.166667
3,reviews_before_and_after_delivery,105,0.057143


In [23]:
# STEP 51-1：レビュー日時の欠損状態を詳しく確認

df_timing_model["missing_review_timestamp_count"] = df_timing_model[
    "missing_review_timestamp_count"
].astype("int64")

missing_timing = df_timing_model.loc[
    df_timing_model["timing_category"] == "review_timestamp_missing"
].copy()

print("=== レビュー日時欠損の内訳 ===")

display(
    missing_timing.groupby(["review_count", "timestamp_count"], dropna=False)
    .agg(
        order_count=("order_id", "size"),
        low_review_rate=("low_review", "mean"),
    )
    .reset_index()
    .sort_values(["review_count", "timestamp_count"])
)

print("\n=== 日時が一部だけ存在する注文 ===")

partial_timing = missing_timing.loc[missing_timing["timestamp_count"] > 0]

print("件数:", len(partial_timing))

display(
    partial_timing[
        [
            "order_id",
            "review_count",
            "timestamp_count",
            "first_review_timestamp",
            "last_review_timestamp",
            "delivered_timestamp",
            "low_review",
        ]
    ].head(10)
)

print("\n=== 暫定予測対象の確認 ===")

df_timing_model["prediction_eligible_candidate"] = (
    df_timing_model["timing_category"] == "all_reviews_after_delivery"
)

display(
    df_timing_model.groupby("prediction_eligible_candidate").agg(
        order_count=("order_id", "size"),
        low_review_count=("low_review", "sum"),
        low_review_rate=("low_review", "mean"),
    )
)


=== レビュー日時欠損の内訳 ===


,review_count,timestamp_count,order_count,low_review_rate
0,1,0,3604,0.290788
1,2,0,9,0.000000
2,2,1,32,0.218750
3,3,2,1,0.000000



=== 日時が一部だけ存在する注文 ===
件数: 33


,order_id,review_count,timestamp_count,first_review_timestamp,last_review_timestamp,delivered_timestamp,low_review
1888,34de5f7952fbd04a9a1e5d7001b014b5,2,1,2018-03-10 00:46:40,2018-03-10 00:46:40,2018-03-08 21:18:31,1
4885,8681822ccf2aaea9124f49a2a6dba7b1,2,1,2018-03-26 21:15:33,2018-03-26 21:15:33,2018-03-14 19:58:50,1
9401,075a544c5f4ed4bb75f82b160465fe76,2,1,2018-03-07 12:32:38,2018-03-07 12:32:38,2018-03-05 20:58:53,1
9802,29f5dd847b7db645b7ce75cc49ddb612,2,1,2018-05-27 14:26:51,2018-05-27 14:26:51,2018-05-07 19:36:52,1
10448,6401cfe160d72392195bd6e311a582cd,2,1,2018-03-22 21:07:04,2018-03-22 21:07:04,2018-03-24 17:02:43,1
11240,ab71e5dff0ed652f1559e1860391bd22,2,1,2018-03-07 12:32:38,2018-03-07 12:32:38,2018-03-22 16:27:35,1
12173,f9fe6e11e33871e71f921d5051e72bb4,2,1,2018-03-19 11:35:24,2018-03-19 11:35:24,2018-03-17 17:08:21,1
12353,03c939fd7fd3b38f8485a0f95798f1f6,3,2,2018-03-06 19:50:32,2018-03-30 00:29:09,2018-03-19 21:48:52,0
13497,29062384ce4975f78aeba6a496510386,2,1,2018-03-14 18:21:23,2018-03-14 18:21:23,2018-03-14 16:58:50,0
13665,2e84df8fa5a2f0d5eab815d6c157b1f3,2,1,2017-10-10 22:47:23,2017-10-10 22:47:23,2017-10-24 10:38:37,0



=== 暫定予測対象の確認 ===


,order_count,low_review_count,low_review_rate
prediction_eligible_candidate,,,
False,8141,4478,0.550055
True,87683,7758,0.088478


In [24]:
# STEP 51-2：各Foldの学習開始時点で利用可能なラベルを確認

fold_boundaries = [
    ("Fold 1", "2017-04-01"),
    ("Fold 2", "2017-07-01"),
    ("Fold 3", "2017-10-01"),
    ("Fold 4", "2018-01-01"),
]

df_eligible = df_timing_model.loc[
    df_timing_model["prediction_eligible_candidate"]
].copy()

results = []

for fold_name, boundary_str in fold_boundaries:
    boundary = pd.Timestamp(boundary_str)

    # 分割境界より前に配送完了した注文
    train_candidates = df_eligible.loc[df_eligible["delivered_timestamp"] < boundary]

    # 分割境界までに全レビューの回答が完了していた注文
    label_available = train_candidates["last_review_timestamp"] < boundary

    results.append(
        {
            "fold": fold_name,
            "training_boundary": boundary_str,
            "candidate_orders": len(train_candidates),
            "label_available_orders": int(label_available.sum()),
            "label_not_yet_available": int((~label_available).sum()),
            "label_available_rate": (
                label_available.mean() if len(train_candidates) > 0 else float("nan")
            ),
            "low_review_rate_available": (
                train_candidates.loc[label_available, "low_review"].mean()
            ),
        }
    )

df_label_availability = pd.DataFrame(results)

print("=== 学習境界時点のラベル利用可能性 ===")
display(df_label_availability)


=== 学習境界時点のラベル利用可能性 ===


,fold,training_boundary,candidate_orders,label_available_orders,label_not_yet_available,label_available_rate,low_review_rate_available
0,Fold 1,2017-04-01,4025,3711,314,0.921988,0.089733
1,Fold 2,2017-07-01,12212,11802,410,0.966426,0.091002
2,Fold 3,2017-10-01,23136,22667,469,0.979729,0.085013
3,Fold 4,2018-01-01,38127,37537,590,0.984525,0.086635


In [25]:
import pandas as pd

# 配送完了後に全レビューが回答された注文
df_cv_base = df_timing_model.loc[
    df_timing_model["timing_category"] == "all_reviews_after_delivery"
].copy()

# 配送完了日時・最終レビュー回答日時があることを確認
assert df_cv_base["delivered_timestamp"].notna().all()
assert df_cv_base["last_review_timestamp"].notna().all()
assert df_cv_base["order_id"].is_unique

fold_configs = [
    ("Fold 1", "2017-04-01", "2017-07-01"),
    ("Fold 2", "2017-07-01", "2017-10-01"),
    ("Fold 3", "2017-10-01", "2018-01-01"),
    ("Fold 4", "2018-01-01", "2018-05-01"),
]

fold_splits = {}
fold_results = []

for fold_name, val_start_str, val_end_str in fold_configs:
    val_start = pd.Timestamp(val_start_str)
    val_end = pd.Timestamp(val_end_str)

    # 学習：検証開始前に配送され、ラベルも確定している注文
    train_mask = (df_cv_base["delivered_timestamp"] < val_start) & (
        df_cv_base["last_review_timestamp"] < val_start
    )

    # 検証：配送完了日時が検証期間内の注文
    val_mask = (df_cv_base["delivered_timestamp"] >= val_start) & (
        df_cv_base["delivered_timestamp"] < val_end
    )

    df_train = df_cv_base.loc[train_mask].copy()
    df_val = df_cv_base.loc[val_mask].copy()

    assert set(df_train["order_id"]).isdisjoint(set(df_val["order_id"]))

    fold_splits[fold_name] = {
        "train_ids": df_train["order_id"].tolist(),
        "val_ids": df_val["order_id"].tolist(),
    }

    fold_results.append(
        {
            "fold": fold_name,
            "train_until": val_start_str,
            "validation_period": (f"{val_start_str} ～ {val_end_str}"),
            "train_orders": len(df_train),
            "val_orders": len(df_val),
            "train_low_review_rate": df_train["low_review"].mean(),
            "val_low_review_rate": df_val["low_review"].mean(),
            "train_max_delivered": df_train["delivered_timestamp"].max(),
            "train_max_review": df_train["last_review_timestamp"].max(),
            "val_min_delivered": df_val["delivered_timestamp"].min(),
            "val_max_delivered": df_val["delivered_timestamp"].max(),
        }
    )

df_fold_summary = pd.DataFrame(fold_results)

print("=== 配送完了日時基準・4-Fold CV ===")
display(df_fold_summary)


=== 配送完了日時基準・4-Fold CV ===


,fold,train_until,validation_period,train_orders,val_orders,train_low_review_rate,val_low_review_rate,train_max_delivered,train_max_review,val_min_delivered,val_max_delivered
0,Fold 1,2017-04-01,2017-04-01 ～ 2017-07-01,3711,8187,0.089733,0.091242,2017-03-30 14:05:20,2017-03-31 23:32:31,2017-04-01 07:38:01,2017-06-30 19:09:42
1,Fold 2,2017-07-01,2017-07-01 ～ 2017-10-01,11802,10924,0.091002,0.079000,2017-06-29 16:32:57,2017-06-30 23:58:51,2017-07-01 07:36:19,2017-09-30 20:09:53
2,Fold 3,2017-10-01,2017-10-01 ～ 2018-01-01,22667,14991,0.085013,0.089320,2017-09-29 20:19:57,2017-09-30 23:59:53,2017-10-01 11:52:06,2017-12-31 15:19:16
3,Fold 4,2018-01-01,2018-01-01 ～ 2018-05-01,37537,23557,0.086635,0.100777,2017-12-30 18:41:57,2017-12-31 23:27:08,2018-01-01 05:47:11,2018-04-30 23:58:51


In [26]:
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

from src.model_metrics import threshold_metrics
from src.model_training import train_logistic_model

# Model Aの特徴量
numeric_features_a = [
    "total_price",
    "total_freight",
    "delivery_delay_days",
    "log_freight_ratio",
    "item_count",
    "category_count",
    "month_sin",
    "month_cos",
]

categorical_features_a = ["customer_state"]

# モデルデータと配送・レビュー日時を結合
df_cv_model = df_model.merge(
    df_cv_base[["order_id", "delivered_timestamp", "last_review_timestamp"]],
    on="order_id",
    how="inner",
    validate="one_to_one",
)

assert len(df_cv_model) == 87683

cv_predictions = []
cv_results = []

for fold_name, start, end in fold_configs:
    start, end = pd.Timestamp(start), pd.Timestamp(end)

    train = df_cv_model.loc[
        (df_cv_model["delivered_timestamp"] < start)
        & (df_cv_model["last_review_timestamp"] < start)
    ]

    val = df_cv_model.loc[
        (df_cv_model["delivered_timestamp"] >= start)
        & (df_cv_model["delivered_timestamp"] < end)
    ]

    model, y_val, probabilities = train_logistic_model(
        train,
        val,
        numeric_features_a,
        categorical_features_a,
    )

    cv_predictions.append(
        pd.DataFrame(
            {
                "fold": fold_name,
                "order_id": val["order_id"].to_numpy(),
                "low_review": y_val.to_numpy(),
                "probability": probabilities,
            }
        )
    )

    metrics = threshold_metrics(y_val, probabilities, threshold=0.24)

    cv_results.append(
        {
            "fold": fold_name,
            "train_orders": len(train),
            "val_orders": len(val),
            "val_low_review_rate": y_val.mean(),
            "roc_auc": roc_auc_score(y_val, probabilities),
            "average_precision": average_precision_score(y_val, probabilities),
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1": metrics["f1"],
        }
    )

df_cv_results = pd.DataFrame(cv_results)

print("=== Model A：配送完了日時基準4-Fold CV ===")
display(df_cv_results)

print("\n=== 4-Foldの平均・標準偏差 ===")
display(
    df_cv_results[
        [
            "roc_auc",
            "average_precision",
            "precision",
            "recall",
            "f1",
        ]
    ].agg(["mean", "std"])
)


=== Model A：配送完了日時基準4-Fold CV ===


,fold,train_orders,val_orders,val_low_review_rate,roc_auc,average_precision,precision,recall,f1
0,Fold 1,3711,8187,0.091242,0.578630,0.132041,0.309524,0.017403,0.032953
1,Fold 2,11802,10924,0.079000,0.562999,0.112763,0.155203,0.101970,0.123077
2,Fold 3,22667,14991,0.089320,0.630262,0.183784,0.372385,0.066468,0.112801
3,Fold 4,37537,23557,0.100777,0.643905,0.212996,0.413551,0.074558,0.126338



=== 4-Foldの平均・標準偏差 ===


,roc_auc,average_precision,precision,recall,f1
mean,0.603949,0.160396,0.312666,0.065100,0.098792
std,0.039187,0.046140,0.113356,0.035241,0.044270


In [27]:
from src.model_metrics import threshold_table

# 各Foldの予測結果を結合
df_cv_predictions = pd.concat(
    cv_predictions,
    ignore_index=True,
)

# 過去の基準しきい値
baseline_threshold = 0.24

# Foldごとにしきい値を比較
for fold_name, group in df_cv_predictions.groupby("fold", sort=True):
    threshold_results = threshold_table(
        group["low_review"],
        group["probability"],
    )

    # F1が高い上位10件のみ表示
    top10 = threshold_results.sort_values(
        ["f1", "threshold"],
        ascending=[False, True],
    ).head(10)

    print(f"\n=== {fold_name}：F1上位10件 ===")
    display(top10)



=== Fold 1：F1上位10件 ===


,threshold,accuracy,precision,recall,f1
6,0.07,0.756687,0.138292,0.318608,0.192869
5,0.06,0.615977,0.114754,0.477912,0.185070
4,0.05,0.417369,0.103645,0.704150,0.180694
7,0.08,0.823256,0.151394,0.203481,0.173615
3,0.04,0.237327,0.095868,0.872825,0.172761
2,0.03,0.151215,0.092510,0.942436,0.168482
1,0.02,0.110419,0.091704,0.982597,0.167752
0,0.01,0.092342,0.091243,0.998661,0.167208
8,0.09,0.859045,0.174400,0.145917,0.158892
9,0.10,0.875290,0.172249,0.096386,0.123605



=== Fold 2：F1上位10件 ===


,threshold,accuracy,precision,recall,f1
13,0.14,0.508696,0.093061,0.596756,0.161013
14,0.15,0.581838,0.094550,0.500579,0.159057
16,0.17,0.701117,0.102054,0.356895,0.158722
15,0.16,0.646558,0.097692,0.421784,0.158640
17,0.18,0.751465,0.107959,0.295481,0.158140
12,0.13,0.426034,0.088946,0.677868,0.157258
19,0.20,0.825613,0.126791,0.205098,0.156707
18,0.19,0.792384,0.115068,0.243337,0.156250
11,0.12,0.337605,0.084388,0.749710,0.151700
10,0.11,0.251556,0.082734,0.840093,0.150634



=== Fold 3：F1上位10件 ===


,threshold,accuracy,precision,recall,f1
11,0.12,0.847175,0.220985,0.281553,0.247619
10,0.11,0.804216,0.183584,0.345780,0.239834
12,0.13,0.870522,0.251240,0.227035,0.238525
13,0.14,0.885998,0.289294,0.189694,0.229138
9,0.10,0.733974,0.154268,0.441374,0.228627
8,0.09,0.611367,0.127387,0.572816,0.208424
14,0.15,0.893536,0.309064,0.155340,0.206759
15,0.16,0.897605,0.325000,0.135922,0.191680
7,0.08,0.389967,0.104158,0.766990,0.183409
16,0.17,0.900540,0.341667,0.122479,0.180319



=== Fold 4：F1上位10件 ===


,threshold,accuracy,precision,recall,f1
11,0.12,0.837713,0.249221,0.303286,0.273608
10,0.11,0.798489,0.212642,0.369840,0.270029
12,0.13,0.860678,0.283603,0.250632,0.266100
9,0.10,0.726069,0.176013,0.466723,0.255623
13,0.14,0.876640,0.320027,0.199242,0.245587
8,0.09,0.578894,0.140256,0.619629,0.228736
14,0.15,0.884153,0.346054,0.168071,0.226255
15,0.16,0.888611,0.368697,0.147852,0.211064
16,0.17,0.891073,0.384892,0.135215,0.200125
7,0.08,0.316212,0.112690,0.841618,0.198766


In [28]:
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

from src.model_training import train_logistic_model

# STEP 51-1：予測対象集団による性能比較

# 既存のModel Aの特徴量を再利用
numeric_features = numeric_features_a
categorical_features = categorical_features_a

# 日時分類情報をモデルデータへ追加（元の行数は維持）
df_compare = df_model.merge(
    df_timing_model[["order_id", "timing_category"]],
    on="order_id",
    how="left",
    validate="one_to_one",
)

assert len(df_compare) == 95824
assert df_compare["order_id"].is_unique
assert df_compare["timing_category"].notna().all()

# 購入月を比較用に統一
purchase_month = pd.to_datetime(
    df_compare["purchase_month"].astype(str),
    format="%Y-%m",
    errors="raise",
)

df_compare["purchase_month_date"] = purchase_month

conditions = {
    "A_all_orders": df_compare,
    "B_after_delivery": df_compare.loc[
        df_compare["timing_category"] == "all_reviews_after_delivery"
    ],
}

results = []

for condition_name, data in conditions.items():
    train = data.loc[data["purchase_month_date"] < pd.Timestamp("2018-01-01")]

    val = data.loc[
        (data["purchase_month_date"] >= pd.Timestamp("2018-01-01"))
        & (data["purchase_month_date"] < pd.Timestamp("2018-05-01"))
    ]

    model, y_val, probabilities = train_logistic_model(
        train,
        val,
        numeric_features,
        categorical_features,
    )

    results.append(
        {
            "condition": condition_name,
            "train_orders": len(train),
            "val_orders": len(val),
            "train_low_review_rate": train["low_review"].mean(),
            "val_low_review_rate": val["low_review"].mean(),
            "roc_auc": roc_auc_score(y_val, probabilities),
            "average_precision": average_precision_score(y_val, probabilities),
        }
    )

df_comparison_ab = pd.DataFrame(results)

print("=== STEP 51-1：予測対象集団の変更による比較 ===")
display(df_comparison_ab)

print("\n=== 実験B − 実験A ===")
print(
    "ROC-AUC差:",
    df_comparison_ab.loc[1, "roc_auc"] - df_comparison_ab.loc[0, "roc_auc"],
)
print(
    "AP差:",
    df_comparison_ab.loc[1, "average_precision"]
    - df_comparison_ab.loc[0, "average_precision"],
)


=== STEP 51-1：予測対象集団の変更による比較 ===


,condition,train_orders,val_orders,train_low_review_rate,val_low_review_rate,roc_auc,average_precision
0,A_all_orders,43362,27220,0.120820,0.164328,0.766418,0.520047
1,B_after_delivery,39968,24009,0.088646,0.098296,0.647648,0.207681



=== 実験B − 実験A ===
ROC-AUC差: -0.11877065236052697
AP差: -0.3123666356915661


In [29]:
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

from src.model_training import train_logistic_model

# 実験C：配送完了日時で分割、ラベル確定制約なし
start = pd.Timestamp("2018-01-01")
end = pd.Timestamp("2018-05-01")

train_c = df_cv_model.loc[df_cv_model["delivered_timestamp"] < start]

val_c = df_cv_model.loc[
    (df_cv_model["delivered_timestamp"] >= start)
    & (df_cv_model["delivered_timestamp"] < end)
]

model_c, y_val_c, proba_c = train_logistic_model(
    train_c,
    val_c,
    numeric_features_a,
    categorical_features_a,
)

result_c = pd.DataFrame(
    [
        {
            "condition": "C_delivery_date",
            "train_orders": len(train_c),
            "val_orders": len(val_c),
            "train_low_review_rate": train_c["low_review"].mean(),
            "val_low_review_rate": y_val_c.mean(),
            "roc_auc": roc_auc_score(y_val_c, proba_c),
            "average_precision": average_precision_score(y_val_c, proba_c),
        }
    ]
)

# A・B・Cを比較
df_comparison_abc = pd.concat(
    [df_comparison_ab, result_c],
    ignore_index=True,
)

print("=== STEP 51：実験A・B・Cの比較 ===")
display(df_comparison_abc.round(4))


=== STEP 51：実験A・B・Cの比較 ===


,condition,train_orders,val_orders,train_low_review_rate,val_low_review_rate,roc_auc,average_precision
0,A_all_orders,43362,27220,0.1208,0.1643,0.7664,0.5200
1,B_after_delivery,39968,24009,0.0886,0.0983,0.6476,0.2077
2,C_delivery_date,38127,23557,0.0868,0.1008,0.6439,0.2132


In [30]:
# STEP 51-3：実験A〜Dの比較表を完成させる

fold4 = df_cv_results.loc[df_cv_results["fold"] == "Fold 4"].iloc[0]

result_d = pd.DataFrame(
    [
        {
            "condition": "D_label_available",
            "train_orders": fold4["train_orders"],
            "val_orders": fold4["val_orders"],
            "train_low_review_rate": 0.086635,
            "val_low_review_rate": fold4["val_low_review_rate"],
            "roc_auc": fold4["roc_auc"],
            "average_precision": fold4["average_precision"],
        }
    ]
)

df_comparison_abcd = pd.concat(
    [df_comparison_abc, result_d],
    ignore_index=True,
)

print("=== STEP 51：実験A〜Dの比較 ===")
display(df_comparison_abcd.round(6))

print("\n=== 条件変更ごとの性能差 ===")

metrics = ["roc_auc", "average_precision"]

differences = df_comparison_abcd[metrics].diff().iloc[1:].copy()

differences.insert(
    0,
    "comparison",
    ["A → B", "B → C", "C → D"],
)

display(differences.round(6))


=== STEP 51：実験A〜Dの比較 ===


,condition,train_orders,val_orders,train_low_review_rate,val_low_review_rate,roc_auc,average_precision
0,A_all_orders,43362,27220,0.120820,0.164328,0.766418,0.520047
1,B_after_delivery,39968,24009,0.088646,0.098296,0.647648,0.207681
2,C_delivery_date,38127,23557,0.086841,0.100777,0.643874,0.213169
3,D_label_available,37537,23557,0.086635,0.100777,0.643905,0.212996



=== 条件変更ごとの性能差 ===


,comparison,roc_auc,average_precision
1,A → B,-0.118771,-0.312367
2,B → C,-0.003774,0.005488
3,C → D,0.000031,-0.000172


今回の実験の結果から、性能低下の大部分が予測対象集団の変更とともに所持ていることを確認した。A → B（予測対象を配送完了してからレビュー投稿した群へ変更）が最も影響が強かった。以前のAには、配送完了前にレビュー投稿されたものや、レビュー日時が不明なものが含まれており、それらの低評価率が高かったことが一因として考えられる。

In [31]:
""" ランダムフォレスト """

import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

from src.model_training import train_random_forest_model

rf_results = []

for fold_name, start, end in fold_configs:
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)

    train = df_cv_model.loc[
        (df_cv_model["delivered_timestamp"] < start)
        & (df_cv_model["last_review_timestamp"] < start)
    ]

    val = df_cv_model.loc[
        (df_cv_model["delivered_timestamp"] >= start)
        & (df_cv_model["delivered_timestamp"] < end)
    ]

    model, y_val, probabilities = train_random_forest_model(
        train,
        val,
        numeric_features_a,
        categorical_features_a,
    )

    rf_results.append(
        {
            "fold": fold_name,
            "train_orders": len(train),
            "val_orders": len(val),
            "roc_auc": roc_auc_score(y_val, probabilities),
            "average_precision": average_precision_score(y_val, probabilities),
        }
    )

df_rf_results = pd.DataFrame(rf_results)

print("=== Random Forest：4-Fold CV ===")
display(df_rf_results)

print("\n=== ロジスティック回帰との比較 ===")

df_model_comparison = df_cv_results[["fold", "roc_auc", "average_precision"]].merge(
    df_rf_results[["fold", "roc_auc", "average_precision"]],
    on="fold",
    suffixes=("_logistic", "_rf"),
    validate="one_to_one",
)

df_model_comparison["auc_diff"] = (
    df_model_comparison["roc_auc_rf"] - df_model_comparison["roc_auc_logistic"]
)

df_model_comparison["ap_diff"] = (
    df_model_comparison["average_precision_rf"]
    - df_model_comparison["average_precision_logistic"]
)

display(df_model_comparison.round(4))

print("\n=== 平均性能 ===")
display(df_model_comparison.drop(columns="fold").mean().to_frame("mean").T.round(4))


=== Random Forest：4-Fold CV ===


,fold,train_orders,val_orders,roc_auc,average_precision
0,Fold 1,3711,8187,0.580711,0.132966
1,Fold 2,11802,10924,0.586830,0.118101
2,Fold 3,22667,14991,0.604469,0.154829
3,Fold 4,37537,23557,0.631745,0.196741



=== ロジスティック回帰との比較 ===


,fold,roc_auc_logistic,average_precision_logistic,roc_auc_rf,average_precision_rf,auc_diff,ap_diff
0,Fold 1,0.5786,0.1320,0.5807,0.1330,0.0021,0.0009
1,Fold 2,0.5630,0.1128,0.5868,0.1181,0.0238,0.0053
2,Fold 3,0.6303,0.1838,0.6045,0.1548,-0.0258,-0.0290
3,Fold 4,0.6439,0.2130,0.6317,0.1967,-0.0122,-0.0163



=== 平均性能 ===


,roc_auc_logistic,average_precision_logistic,roc_auc_rf,average_precision_rf,auc_diff,ap_diff
mean,0.6039,0.1604,0.6009,0.1507,-0.003,-0.0097


In [32]:
""" XGBoost """

import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score
from xgboost import XGBClassifier

from src.model_training import train_classifier_model

xgb_results = []

for fold_name, start, end in fold_configs:
    start, end = pd.Timestamp(start), pd.Timestamp(end)

    train = df_cv_model.loc[
        (df_cv_model["delivered_timestamp"] < start)
        & (df_cv_model["last_review_timestamp"] < start)
    ]

    val = df_cv_model.loc[
        (df_cv_model["delivered_timestamp"] >= start)
        & (df_cv_model["delivered_timestamp"] < end)
    ]

    classifier = XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
    )

    model, y_val, probabilities = train_classifier_model(
        train,
        val,
        numeric_features_a,
        categorical_features_a,
        classifier,
    )

    xgb_results.append(
        {
            "fold": fold_name,
            "train_orders": len(train),
            "val_orders": len(val),
            "roc_auc": roc_auc_score(y_val, probabilities),
            "average_precision": average_precision_score(y_val, probabilities),
        }
    )

df_xgb_results = pd.DataFrame(xgb_results)

print("=== XGBoost：4-Fold CV ===")
display(df_xgb_results.round(4))

print("\n=== 3モデルのFold別比較 ===")

df_three_models = pd.concat(
    [
        df_cv_results.assign(model="Logistic Regression"),
        df_rf_results.assign(model="Random Forest"),
        df_xgb_results.assign(model="XGBoost"),
    ],
    ignore_index=True,
)

display(
    df_three_models.pivot(
        index="fold",
        columns="model",
        values=["roc_auc", "average_precision"],
    ).round(4)
)

print("\n=== 3モデルの平均性能 ===")
display(
    df_three_models.groupby("model")[["roc_auc", "average_precision"]]
    .agg(["mean", "std"])
    .round(4)
)


=== XGBoost：4-Fold CV ===


,fold,train_orders,val_orders,roc_auc,average_precision
0,Fold 1,3711,8187,0.5719,0.1239
1,Fold 2,11802,10924,0.5678,0.1108
2,Fold 3,22667,14991,0.6159,0.1467
3,Fold 4,37537,23557,0.6489,0.2110



=== 3モデルのFold別比較 ===


roc_auc                         average_precision  \
model  Logistic Regression Random Forest XGBoost Logistic Regression   
fold                                                                   
Fold 1              0.5786        0.5807  0.5719              0.1320   
Fold 2              0.5630        0.5868  0.5678              0.1128   
Fold 3              0.6303        0.6045  0.6159              0.1838   
Fold 4              0.6439        0.6317  0.6489              0.2130   

                              
model  Random Forest XGBoost  
fold                          
Fold 1        0.1330  0.1239  
Fold 2        0.1181  0.1108  
Fold 3        0.1548  0.1467  
Fold 4        0.1967  0.2110


=== 3モデルの平均性能 ===


roc_auc         average_precision        
                       mean     std              mean     std
model                                                        
Logistic Regression  0.6039  0.0392            0.1604  0.0461
Random Forest        0.6009  0.0229            0.1507  0.0342
XGBoost              0.6012  0.0386            0.1481  0.0445

In [33]:
""" LightGBM """

import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

from src.model_training import train_classifier_model

lgbm_results = []

for fold_name, start, end in fold_configs:
    start, end = pd.Timestamp(start), pd.Timestamp(end)

    train = df_cv_model.loc[
        (df_cv_model["delivered_timestamp"] < start)
        & (df_cv_model["last_review_timestamp"] < start)
    ]

    val = df_cv_model.loc[
        (df_cv_model["delivered_timestamp"] >= start)
        & (df_cv_model["delivered_timestamp"] < end)
    ]

    classifier = LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=15,
        max_depth=5,
        min_child_samples=30,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
    )

    model, y_val, probabilities = train_classifier_model(
        train,
        val,
        numeric_features_a,
        categorical_features_a,
        classifier,
    )

    lgbm_results.append(
        {
            "fold": fold_name,
            "train_orders": len(train),
            "val_orders": len(val),
            "roc_auc": roc_auc_score(y_val, probabilities),
            "average_precision": average_precision_score(y_val, probabilities),
        }
    )

df_lgbm_results = pd.DataFrame(lgbm_results)

print("=== LightGBM：4-Fold CV ===")
display(df_lgbm_results.round(4))


=== LightGBM：4-Fold CV ===


,fold,train_orders,val_orders,roc_auc,average_precision
0,Fold 1,3711,8187,0.5546,0.1184
1,Fold 2,11802,10924,0.5692,0.1131
2,Fold 3,22667,14991,0.5952,0.1337
3,Fold 4,37537,23557,0.6418,0.1986


In [34]:
df_four_models = pd.concat(
    [
        df_cv_results.assign(model="Logistic Regression"),
        df_rf_results.assign(model="Random Forest"),
        df_xgb_results.assign(model="XGBoost"),
        df_lgbm_results.assign(model="LightGBM"),
    ],
    ignore_index=True,
)

print("=== 4モデルのFold別性能 ===")
display(
    df_four_models.pivot(
        index="fold",
        columns="model",
        values=["roc_auc", "average_precision"],
    ).round(4)
)

print("\n=== 4モデルの平均・標準偏差 ===")
display(
    df_four_models.groupby("model")[["roc_auc", "average_precision"]]
    .agg(["mean", "std"])
    .round(4)
)


=== 4モデルのFold別性能 ===


roc_auc                                           average_precision  \
model  LightGBM Logistic Regression Random Forest XGBoost          LightGBM   
fold                                                                          
Fold 1   0.5546              0.5786        0.5807  0.5719            0.1184   
Fold 2   0.5692              0.5630        0.5868  0.5678            0.1131   
Fold 3   0.5952              0.6303        0.6045  0.6159            0.1337   
Fold 4   0.6418              0.6439        0.6317  0.6489            0.1986   

                                                  
model  Logistic Regression Random Forest XGBoost  
fold                                              
Fold 1              0.1320        0.1330  0.1239  
Fold 2              0.1128        0.1181  0.1108  
Fold 3              0.1838        0.1548  0.1467  
Fold 4              0.2130        0.1967  0.2110


=== 4モデルの平均・標準偏差 ===


roc_auc         average_precision        
                       mean     std              mean     std
model                                                        
LightGBM             0.5902  0.0383            0.1410  0.0394
Logistic Regression  0.6039  0.0392            0.1604  0.0461
Random Forest        0.6009  0.0229            0.1507  0.0342
XGBoost              0.6012  0.0386            0.1481  0.0445

In [35]:
""" CatBoost """

import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

from src.model_training import train_classifier_model

cat_results = []

for fold_name, start, end in fold_configs:
    start, end = pd.Timestamp(start), pd.Timestamp(end)

    train = df_cv_model.loc[
        (df_cv_model["delivered_timestamp"] < start)
        & (df_cv_model["last_review_timestamp"] < start)
    ]

    val = df_cv_model.loc[
        (df_cv_model["delivered_timestamp"] >= start)
        & (df_cv_model["delivered_timestamp"] < end)
    ]

    classifier = CatBoostClassifier(
        iterations=300,
        depth=4,
        learning_rate=0.05,
        loss_function="Logloss",
        random_seed=42,
        task_type="CPU",
        thread_count=-1,
        verbose=False,
        allow_writing_files=False,
    )

    model, y_val, probabilities = train_classifier_model(
        train,
        val,
        numeric_features_a,
        categorical_features_a,
        classifier,
    )

    cat_results.append(
        {
            "fold": fold_name,
            "train_orders": len(train),
            "val_orders": len(val),
            "roc_auc": roc_auc_score(y_val, probabilities),
            "average_precision": average_precision_score(y_val, probabilities),
        }
    )

df_cat_results = pd.DataFrame(cat_results)

print("=== CatBoost：4-Fold CV ===")
display(df_cat_results.round(4))


=== CatBoost：4-Fold CV ===


,fold,train_orders,val_orders,roc_auc,average_precision
0,Fold 1,3711,8187,0.5692,0.1285
1,Fold 2,11802,10924,0.5682,0.1078
2,Fold 3,22667,14991,0.6199,0.1402
3,Fold 4,37537,23557,0.6490,0.2167


In [36]:
# これまでの評価結果を結合
df_five_models = pd.concat(
    [
        df_cv_results.assign(model="Logistic Regression"),
        df_rf_results.assign(model="Random Forest"),
        df_xgb_results.assign(model="XGBoost"),
        df_lgbm_results.assign(model="LightGBM"),
        df_cat_results.assign(model="CatBoost"),
    ],
    ignore_index=True,
)

print("=== 5モデルのFold別性能 ===")
display(
    df_five_models.pivot(
        index="fold",
        columns="model",
        values=["roc_auc", "average_precision"],
    ).round(4)
)

print("\n=== 5モデルの平均・標準偏差 ===")
display(
    df_five_models.groupby("model")[["roc_auc", "average_precision"]]
    .agg(["mean", "std"])
    .round(4)
    .sort_values(("roc_auc", "mean"), ascending=False)
)


=== 5モデルのFold別性能 ===


roc_auc                                                     \
model  CatBoost LightGBM Logistic Regression Random Forest XGBoost   
fold                                                                 
Fold 1   0.5692   0.5546              0.5786        0.5807  0.5719   
Fold 2   0.5682   0.5692              0.5630        0.5868  0.5678   
Fold 3   0.6199   0.5952              0.6303        0.6045  0.6159   
Fold 4   0.6490   0.6418              0.6439        0.6317  0.6489   

       average_precision                                                     
model           CatBoost LightGBM Logistic Regression Random Forest XGBoost  
fold                                                                         
Fold 1            0.1285   0.1184              0.1320        0.1330  0.1239  
Fold 2            0.1078   0.1131              0.1128        0.1181  0.1108  
Fold 3            0.1402   0.1337              0.1838        0.1548  0.1467  
Fold 4            0.2167   0.1986              0.2130        0.1967  0.2110


=== 5モデルの平均・標準偏差 ===


roc_auc         average_precision        
                       mean     std              mean     std
model                                                        
Logistic Regression  0.6039  0.0392            0.1604  0.0461
CatBoost             0.6016  0.0398            0.1483  0.0475
XGBoost              0.6012  0.0386            0.1481  0.0445
Random Forest        0.6009  0.0229            0.1507  0.0342
LightGBM             0.5902  0.0383            0.1410  0.0394

ここまで、特徴量Aを土台に検討してきたので、これから特徴量B,C,Dでもそれぞれ検討してみる。

In [37]:
# STEP 52-1：特徴量セットA〜Dの定義

feature_sets = {
    "A_Order": {
        "numeric": numeric_features_a,
        "categorical": categorical_features_a,
    },
    "B_Payment": {
        "numeric": numeric_features_a
        + [
            "payment_count",
            "payment_type_count",
            "max_payment_installments",
        ],
        "categorical": categorical_features_a
        + [
            "main_payment_type",
        ],
    },
    "C_Category": {
        "numeric": numeric_features_a
        + [
            "payment_count",
            "payment_type_count",
            "max_payment_installments",
        ],
        "categorical": categorical_features_a
        + [
            "main_payment_type",
            "main_category",
        ],
    },
    "D_History": {
        "numeric": numeric_features_a
        + [
            "payment_count",
            "payment_type_count",
            "max_payment_installments",
            "prior_order_count",
            "is_repeat_customer",
            "log_prior_customer_gmv",
            "log_prior_avg_order_value",
            "days_since_previous_order",
        ],
        "categorical": categorical_features_a
        + [
            "main_payment_type",
            "main_category",
        ],
    },
}

# 必要な特徴量がすべて存在するか確認
for model_name, features in feature_sets.items():
    columns = features["numeric"] + features["categorical"]

    missing = [column for column in columns if column not in df_cv_model.columns]

    assert not missing, f"{model_name}: 存在しない列 {missing}"

    print(
        f"{model_name}: "
        f"数値{len(features['numeric'])}列、"
        f"カテゴリ{len(features['categorical'])}列"
    )


A_Order: 数値8列、カテゴリ1列
B_Payment: 数値11列、カテゴリ2列
C_Category: 数値11列、カテゴリ3列
D_History: 数値16列、カテゴリ3列


In [38]:
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

from src.model_training import train_logistic_model

feature_cv_results = []

for model_name, features in feature_sets.items():
    for fold_name, start, end in fold_configs:
        start = pd.Timestamp(start)
        end = pd.Timestamp(end)

        # 学習時点で配送とレビュー回答が完了した注文
        train = df_cv_model.loc[
            (df_cv_model["delivered_timestamp"] < start)
            & (df_cv_model["last_review_timestamp"] < start)
        ]

        # 検証期間に配送完了した注文
        val = df_cv_model.loc[
            (df_cv_model["delivered_timestamp"] >= start)
            & (df_cv_model["delivered_timestamp"] < end)
        ]

        # 既存関数で学習・予測
        model, y_val, probabilities = train_logistic_model(
            train,
            val,
            features["numeric"],
            features["categorical"],
        )

        # 評価結果を記録
        feature_cv_results.append(
            {
                "model": model_name,
                "fold": fold_name,
                "train_orders": len(train),
                "val_orders": len(val),
                "roc_auc": roc_auc_score(y_val, probabilities),
                "average_precision": average_precision_score(y_val, probabilities),
            }
        )

df_feature_cv_results = pd.DataFrame(feature_cv_results)

print("=== 特徴量セットA〜D：Fold別性能 ===")

display(
    df_feature_cv_results.pivot(
        index="fold",
        columns="model",
        values=["roc_auc", "average_precision"],
    ).round(4)
)

print("\n=== 特徴量セットA〜D：平均・標準偏差 ===")

display(
    df_feature_cv_results.groupby("model")[["roc_auc", "average_precision"]]
    .agg(["mean", "std"])
    .round(4)
)


=== 特徴量セットA〜D：Fold別性能 ===


roc_auc                                average_precision            \
model  A_Order B_Payment C_Category D_History           A_Order B_Payment   
fold                                                                        
Fold 1  0.5786    0.5762     0.5771    0.5756            0.1320    0.1312   
Fold 2  0.5630    0.5663     0.5656    0.5637            0.1128    0.1135   
Fold 3  0.6303    0.6308     0.6453    0.6473            0.1838    0.1807   
Fold 4  0.6439    0.6449     0.6456    0.6441            0.2130    0.2119   

                             
model  C_Category D_History  
fold                         
Fold 1     0.1266    0.1258  
Fold 2     0.1118    0.1088  
Fold 3     0.1818    0.1825  
Fold 4     0.2060    0.2045


=== 特徴量セットA〜D：平均・標準偏差 ===


roc_auc         average_precision        
              mean     std              mean     std
model                                               
A_Order     0.6039  0.0392            0.1604  0.0461
B_Payment   0.6046  0.0391            0.1593  0.0451
C_Category  0.6084  0.0431            0.1565  0.0447
D_History   0.6077  0.0442            0.1554  0.0454

この結果を見ると、特徴量の追加で大きな変化は見られないが、APはA_Orderが、mean_roc_aucはC_Categoryが最も高い。また、C_Catagoryを追加したことによって、Fold3の予測性能が向上している。これらのことから、A_Order（特徴量が最も少なく、APが高い）、C_Category（mean_roc_aucが最も高く、Dへの変化によってあまり変化していない）を今後の検討データとする。

In [39]:
from catboost import CatBoostClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from xgboost import XGBClassifier

from src.model_training import train_classifier_model

# Model Cの特徴量を取得
features_c = feature_sets["C_Category"]

# 比較する分類器
classifiers = {
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
    ),
    "CatBoost": CatBoostClassifier(
        iterations=300,
        depth=4,
        learning_rate=0.05,
        loss_function="Logloss",
        random_seed=42,
        task_type="CPU",
        thread_count=-1,
        verbose=False,
        allow_writing_files=False,
    ),
}

model_c_results = []

for model_name, classifier in classifiers.items():
    for fold_name, start, end in fold_configs:
        start = pd.Timestamp(start)
        end = pd.Timestamp(end)

        # 学習時点までに配送・レビュー回答が完了
        train = df_cv_model.loc[
            (df_cv_model["delivered_timestamp"] < start)
            & (df_cv_model["last_review_timestamp"] < start)
        ]

        # 検証期間に配送された注文
        val = df_cv_model.loc[
            (df_cv_model["delivered_timestamp"] >= start)
            & (df_cv_model["delivered_timestamp"] < end)
        ]

        # srcの共通関数で学習・予測
        model, y_val, probabilities = train_classifier_model(
            train,
            val,
            features_c["numeric"],
            features_c["categorical"],
            classifier,
        )

        model_c_results.append(
            {
                "model": model_name,
                "fold": fold_name,
                "roc_auc": roc_auc_score(y_val, probabilities),
                "average_precision": average_precision_score(y_val, probabilities),
            }
        )

df_model_c_results = pd.DataFrame(model_c_results)

# 前回のModel Aの結果と比較する
df_ablation_compare = pd.concat(
    [
        df_xgb_results.assign(model="XGBoost", features="A"),
        df_cat_results.assign(model="CatBoost", features="A"),
        df_model_c_results.assign(features="C"),
    ],
    ignore_index=True,
)

print("=== Model A・Cのアルゴリズム別比較 ===")
display(
    df_ablation_compare.groupby(["model", "features"])[["roc_auc", "average_precision"]]
    .agg(["mean", "std"])
    .round(4)
)

print("\n=== Fold別ROC-AUC ===")
display(
    df_ablation_compare.pivot_table(
        index="fold",
        columns=["model", "features"],
        values="roc_auc",
    ).round(4)
)

print("\n=== Fold別AP ===")
display(
    df_ablation_compare.pivot_table(
        index="fold",
        columns=["model", "features"],
        values="average_precision",
    ).round(4)
)


=== Model A・Cのアルゴリズム別比較 ===


roc_auc         average_precision        
                     mean     std              mean     std
model    features                                          
CatBoost A         0.6016  0.0398            0.1483  0.0475
         C         0.6206  0.0380            0.1614  0.0471
XGBoost  A         0.6012  0.0386            0.1481  0.0445
         C         0.6157  0.0440            0.1544  0.0462


=== Fold別ROC-AUC ===


model    CatBoost         XGBoost        
features        A       C       A       C
fold                                     
Fold 1     0.5692  0.5784  0.5719  0.5765
Fold 2     0.5682  0.6001  0.5678  0.5813
Fold 3     0.6199  0.6428  0.6159  0.6385
Fold 4     0.6490  0.6611  0.6489  0.6663


=== Fold別AP ===


model    CatBoost         XGBoost        
features        A       C       A       C
fold                                     
Fold 1     0.1285  0.1294  0.1239  0.1223
Fold 2     0.1078  0.1191  0.1108  0.1180
Fold 3     0.1402  0.1755  0.1467  0.1594
Fold 4     0.2167  0.2216  0.2110  0.2178

この結果をみると、ロジスティック回帰ではA → Cでそこまでの性能向上は見られなかったが、どちらのモデルでも性能が向上していることが確認された。特に、CatBootは性能の向上が大きいので、現時点ではCatBoostを採用する方向で考える。また、現在は特徴量Cでの検証だが、特徴量Bでの検証をすることによって、どの特徴量が性能に効いているのかを検証する。

In [40]:
from sklearn.metrics import average_precision_score, roc_auc_score

from src.model_training import train_classifier_model

features_b = feature_sets["B_Payment"]
cat_b_results = []

for fold_name, start, end in fold_configs:
    start, end = pd.Timestamp(start), pd.Timestamp(end)

    train = df_cv_model.loc[
        (df_cv_model["delivered_timestamp"] < start)
        & (df_cv_model["last_review_timestamp"] < start)
    ]

    val = df_cv_model.loc[
        (df_cv_model["delivered_timestamp"] >= start)
        & (df_cv_model["delivered_timestamp"] < end)
    ]

    model, y_val, probabilities = train_classifier_model(
        train,
        val,
        features_b["numeric"],
        features_b["categorical"],
        classifiers["CatBoost"],
    )

    cat_b_results.append(
        {
            "model": "CatBoost",
            "features": "B",
            "fold": fold_name,
            "roc_auc": roc_auc_score(y_val, probabilities),
            "average_precision": average_precision_score(y_val, probabilities),
        }
    )

df_cat_b_results = pd.DataFrame(cat_b_results)

# 既存のCatBoost A・Cと比較
df_cat_abc = pd.concat(
    [
        df_ablation_compare.loc[df_ablation_compare["model"] == "CatBoost"],
        df_cat_b_results,
    ],
    ignore_index=True,
)

print("=== CatBoost：特徴量セットA・B・Cの平均比較 ===")
display(
    df_cat_abc.groupby("features")[["roc_auc", "average_precision"]]
    .agg(["mean", "std"])
    .round(4)
)

print("\n=== Fold別の比較 ===")
display(
    df_cat_abc.pivot(
        index="fold",
        columns="features",
        values=["roc_auc", "average_precision"],
    ).round(4)
)


=== CatBoost：特徴量セットA・B・Cの平均比較 ===


roc_auc         average_precision        
            mean     std              mean     std
features                                          
A         0.6016  0.0398            0.1483  0.0475
B         0.6047  0.0401            0.1514  0.0468
C         0.6206  0.0380            0.1614  0.0471


=== Fold別の比較 ===


roc_auc                 average_precision                
features       A       B       C                 A       B       C
fold                                                              
Fold 1    0.5692  0.5687  0.5784            0.1285  0.1298  0.1294
Fold 2    0.5682  0.5745  0.6001            0.1078  0.1111  0.1191
Fold 3    0.6199  0.6232  0.6428            0.1402  0.1468  0.1755
Fold 4    0.6490  0.6524  0.6611            0.2167  0.2181  0.2216

A → B → Cとそれぞれ順に性能改善がみられる。特に、B → Cへの性能改善が大きいので、支払情報の追加（B）より、商品カテゴリーの追加（C）が性能改善に寄与している。この調子だと、ロジスティック回帰では改善しなかった、顧客の情報（D）入れても性能が改善する可能性が考えられる。

In [41]:
# STEP 52-6：CatBoost Model D（顧客履歴追加）の評価

features_d = feature_sets["D_History"]

cat_d_results = []

for fold_name, start, end in fold_configs:
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)

    # 学習時点で配送完了・レビュー回答が確定した注文
    train = df_cv_model.loc[
        (df_cv_model["delivered_timestamp"] < start)
        & (df_cv_model["last_review_timestamp"] < start)
    ]

    # 検証期間に配送完了した注文
    val = df_cv_model.loc[
        (df_cv_model["delivered_timestamp"] >= start)
        & (df_cv_model["delivered_timestamp"] < end)
    ]

    model, y_val, probabilities = train_classifier_model(
        train,
        val,
        features_d["numeric"],
        features_d["categorical"],
        classifiers["CatBoost"],
    )

    cat_d_results.append(
        {
            "model": "CatBoost",
            "features": "D",
            "fold": fold_name,
            "roc_auc": roc_auc_score(y_val, probabilities),
            "average_precision": average_precision_score(y_val, probabilities),
        }
    )

df_cat_d_results = pd.DataFrame(cat_d_results)

# 既存のModel A・B・Cと結合
df_cat_abcd = pd.concat(
    [df_cat_abc, df_cat_d_results],
    ignore_index=True,
)

print("=== CatBoost：特徴量セットA〜Dの平均比較 ===")
display(
    df_cat_abcd.groupby("features")[["roc_auc", "average_precision"]]
    .agg(["mean", "std"])
    .round(4)
)

print("\n=== CatBoost：Fold別比較 ===")
display(
    df_cat_abcd.pivot(
        index="fold",
        columns="features",
        values=["roc_auc", "average_precision"],
    ).round(4)
)


=== CatBoost：特徴量セットA〜Dの平均比較 ===


roc_auc         average_precision        
            mean     std              mean     std
features                                          
A         0.6016  0.0398            0.1483  0.0475
B         0.6047  0.0401            0.1514  0.0468
C         0.6206  0.0380            0.1614  0.0471
D         0.6211  0.0410            0.1586  0.0486


=== CatBoost：Fold別比較 ===


roc_auc                         average_precision                  \
features       A       B       C       D                 A       B       C   
fold                                                                         
Fold 1    0.5692  0.5687  0.5784  0.5799            0.1285  0.1298  0.1294   
Fold 2    0.5682  0.5745  0.6001  0.5928            0.1078  0.1111  0.1191   
Fold 3    0.6199  0.6232  0.6428  0.6482            0.1402  0.1468  0.1755   
Fold 4    0.6490  0.6524  0.6611  0.6635            0.2167  0.2181  0.2216   

                  
features       D  
fold              
Fold 1    0.1254  
Fold 2    0.1160  
Fold 3    0.1710  
Fold 4    0.2220

In [44]:
import numpy as np
import pandas as pd

from src.project_paths import DATA_DIR

# 修正後の顧客履歴データを読み込む
df_history_corrected = pd.read_csv(DATA_DIR / "customer_history_features_corrected.csv")

history_features = [
    "prior_order_count",
    "is_repeat_customer",
    "prior_customer_gmv",
    "prior_avg_order_value",
    "days_since_previous_order",
]

print("=== 修正後データの品質確認 ===")
print("行数:", len(df_history_corrected))
print("order_id重複:", df_history_corrected["order_id"].duplicated().sum())

# 元モデルデータと注文IDで結合
df_history_comparison = df_model[["order_id"] + history_features].merge(
    df_history_corrected[["order_id"] + history_features],
    on="order_id",
    how="left",
    suffixes=("_old", "_new"),
    validate="one_to_one",
    indicator=True,
)

print("\n=== モデルデータとの結合 ===")
print(df_history_comparison["_merge"].value_counts())

print("\n=== 修正前後の特徴量変更件数 ===")

comparison_results = []

for feature in history_features:
    old = df_history_comparison[f"{feature}_old"]
    new = df_history_comparison[f"{feature}_new"]

    # 浮動小数点の微小な誤差を許容しつつ、両方NaNなら一致
    same = np.isclose(
        old,
        new,
        rtol=1e-9,
        atol=1e-8,
        equal_nan=True,
    )

    comparison_results.append(
        {
            "feature": feature,
            "changed_orders": int((~same).sum()),
            "change_rate": float((~same).mean()),
        }
    )

display(pd.DataFrame(comparison_results))


=== 修正後データの品質確認 ===
行数: 98666
order_id重複: 0

=== モデルデータとの結合 ===
_merge
both          95824
left_only         0
right_only        0
Name: count, dtype: int64

=== 修正前後の特徴量変更件数 ===


,feature,changed_orders,change_rate
0,prior_order_count,271,0.002828
1,is_repeat_customer,259,0.002703
2,prior_customer_gmv,271,0.002828
3,prior_avg_order_value,267,0.002786
4,days_since_previous_order,92928,0.969778


In [45]:
# 修正前後の「前回購入からの日数」を確認

old_days = df_history_comparison["days_since_previous_order_old"]
new_days = df_history_comparison["days_since_previous_order_new"]

print("=== 欠損値の比較 ===")

print("修正前の欠損数:", old_days.isna().sum())
print("修正後の欠損数:", new_days.isna().sum())

print("\n=== 修正前後の代表的な値 ===")

display(
    pd.DataFrame(
        {
            "old": old_days,
            "new": new_days,
        }
    ).describe()
)

print("\n=== 初回購入注文の値 ===")

first_orders = df_history_comparison["prior_order_count_old"] == 0

display(
    df_history_comparison.loc[
        first_orders,
        [
            "days_since_previous_order_old",
            "days_since_previous_order_new",
        ],
    ].head(10)
)

print("\n=== 両方の値が存在する注文の変更件数 ===")

both_valid = old_days.notna() & new_days.notna()

actual_difference = ~np.isclose(
    old_days.loc[both_valid],
    new_days.loc[both_valid],
    rtol=1e-9,
    atol=1e-8,
)

print("比較可能な注文数:", both_valid.sum())
print("値が異なる注文数:", actual_difference.sum())


=== 欠損値の比較 ===
修正前の欠損数: 0
修正後の欠損数: 92923

=== 修正前後の代表的な値 ===


,old,new
count,95824.000000,2901.000000
mean,2.600413,85.991727
std,24.112185,109.761525
min,0.000000,0.000000
25%,0.000000,2.000000
50%,0.000000,39.000000
75%,0.000000,133.000000
max,608.000000,608.000000



=== 初回購入注文の値 ===


,days_since_previous_order_old,days_since_previous_order_new
0,0.0,NaN
1,0.0,NaN
2,0.0,NaN
3,0.0,NaN
4,0.0,NaN
5,0.0,NaN
6,0.0,NaN
7,0.0,NaN
8,0.0,NaN
9,0.0,NaN



=== 両方の値が存在する注文の変更件数 ===
比較可能な注文数: 2901
値が異なる注文数: 5


In [46]:
# 修正版顧客履歴をモデルデータに反映する

history_features = [
    "prior_order_count",
    "is_repeat_customer",
    "prior_customer_gmv",
    "prior_avg_order_value",
    "days_since_previous_order",
]

# 古い顧客履歴特徴量を除き、修正版を結合
df_model_corrected = df_model.drop(columns=history_features).merge(
    df_history_corrected[["order_id"] + history_features],
    on="order_id",
    how="left",
    validate="one_to_one",
)

# 従来と同じ欠損値処理
df_model_corrected["days_since_previous_order"] = df_model_corrected[
    "days_since_previous_order"
].fillna(0)

# 対数特徴量を修正版から再計算
df_model_corrected["log_prior_customer_gmv"] = np.log1p(
    df_model_corrected["prior_customer_gmv"]
)

df_model_corrected["log_prior_avg_order_value"] = np.log1p(
    df_model_corrected["prior_avg_order_value"]
)

# 配送・レビュー日時を結合してCV対象を再構築
df_cv_model_corrected = df_model_corrected.merge(
    df_cv_base[["order_id", "delivered_timestamp", "last_review_timestamp"]],
    on="order_id",
    how="inner",
    validate="one_to_one",
)

assert len(df_model_corrected) == len(df_model)
assert len(df_cv_model_corrected) == len(df_cv_model)

print("=== 修正版モデルデータ ===")
print("元のモデル:", df_model.shape)
print("修正版モデル:", df_model_corrected.shape)
print("CV対象:", df_cv_model_corrected.shape)

print("\n=== 顧客履歴特徴量の欠損 ===")
display(
    df_cv_model_corrected[
        history_features
        + [
            "log_prior_customer_gmv",
            "log_prior_avg_order_value",
        ]
    ]
    .isna()
    .sum()
)


=== 修正版モデルデータ ===
元のモデル: (95824, 29)
修正版モデル: (95824, 29)
CV対象: (87683, 31)

=== 顧客履歴特徴量の欠損 ===


prior_order_count            0
is_repeat_customer           0
prior_customer_gmv           0
prior_avg_order_value        0
days_since_previous_order    0
log_prior_customer_gmv       0
log_prior_avg_order_value    0
dtype: int64

In [47]:
# STEP 52-9：修正版Model Dの4-Fold CV

import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

from src.model_training import train_classifier_model

features_d = feature_sets["D_History"]

corrected_results = []

for fold_name, start, end in fold_configs:
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)

    # 学習時点で配送・レビュー回答が完了した注文
    train = df_cv_model_corrected.loc[
        (df_cv_model_corrected["delivered_timestamp"] < start)
        & (df_cv_model_corrected["last_review_timestamp"] < start)
    ]

    # 検証期間中に配送完了した注文
    val = df_cv_model_corrected.loc[
        (df_cv_model_corrected["delivered_timestamp"] >= start)
        & (df_cv_model_corrected["delivered_timestamp"] < end)
    ]

    model, y_val, probabilities = train_classifier_model(
        train,
        val,
        features_d["numeric"],
        features_d["categorical"],
        classifiers["CatBoost"],
    )

    corrected_results.append(
        {
            "fold": fold_name,
            "roc_auc": roc_auc_score(y_val, probabilities),
            "average_precision": average_precision_score(y_val, probabilities),
        }
    )

df_cat_d_corrected = pd.DataFrame(corrected_results)

# 修正前Model Dと比較
df_d_comparison = df_cat_d_results[["fold", "roc_auc", "average_precision"]].merge(
    df_cat_d_corrected,
    on="fold",
    suffixes=("_old", "_corrected"),
    validate="one_to_one",
)

df_d_comparison["auc_diff"] = (
    df_d_comparison["roc_auc_corrected"] - df_d_comparison["roc_auc_old"]
)

df_d_comparison["ap_diff"] = (
    df_d_comparison["average_precision_corrected"]
    - df_d_comparison["average_precision_old"]
)

print("=== Model D：顧客履歴修正前後の比較 ===")
display(df_d_comparison.round(6))

print("\n=== 修正版Model Dの平均性能 ===")
display(
    df_cat_d_corrected[["roc_auc", "average_precision"]].agg(["mean", "std"]).round(6)
)

print("\n=== Model Cとの平均性能比較 ===")
df_c_vs_d = pd.concat(
    [
        df_cat_abc.loc[
            df_cat_abc["features"] == "C", ["fold", "roc_auc", "average_precision"]
        ].assign(features="C"),
        df_cat_d_corrected.assign(features="D_corrected"),
    ]
)

display(df_c_vs_d.groupby("features")[["roc_auc", "average_precision"]].mean().round(6))


=== Model D：顧客履歴修正前後の比較 ===


,fold,roc_auc_old,average_precision_old,roc_auc_corrected,average_precision_corrected,auc_diff,ap_diff
0,Fold 1,0.579922,0.125438,0.583066,0.127458,0.003144,0.002019
1,Fold 2,0.592804,0.115983,0.592883,0.118325,0.000079,0.002342
2,Fold 3,0.648195,0.171007,0.647373,0.177528,-0.000822,0.006521
3,Fold 4,0.663549,0.221990,0.663256,0.220683,-0.000293,-0.001307



=== 修正版Model Dの平均性能 ===


,roc_auc,average_precision
mean,0.621645,0.160998
std,0.039619,0.047545



=== Model Cとの平均性能比較 ===


,roc_auc,average_precision
features,,
C,0.620601,0.161399
D_corrected,0.621645,0.160998


顧客の過去の支払方法で、同日に複数注文をしている場合に過去の注文データとして換算されてしまう点を修正し、feature_Dを作成したが、大きな性能改善は見られなかった。fearture_Cを基準として、続いて、特徴量を簡素化するために、A → Cで支払い情報（B）と商品カテゴリー（C）を追加していたが、fearture_Eとして、A + 商品カテゴリーのみで検討してみる。

In [48]:
# STEP 52-10：Model E（基本特徴量＋商品カテゴリ）

feature_sets["E_Category_Only"] = {
    "numeric": numeric_features_a.copy(),
    "categorical": categorical_features_a
    + [
        "main_category",
    ],
}

# 特徴量の確認
features_e = feature_sets["E_Category_Only"]

print("=== Model Eの特徴量 ===")
print("数値特徴量:", features_e["numeric"])
print("カテゴリ特徴量:", features_e["categorical"])

# Model Cとの差分を確認
features_c = feature_sets["C_Category"]

print("\n=== Model Cにのみ存在する特徴量 ===")
print(
    sorted(
        set(features_c["numeric"] + features_c["categorical"])
        - set(features_e["numeric"] + features_e["categorical"])
    )
)


=== Model Eの特徴量 ===
数値特徴量: ['total_price', 'total_freight', 'delivery_delay_days', 'log_freight_ratio', 'item_count', 'category_count', 'month_sin', 'month_cos']
カテゴリ特徴量: ['customer_state', 'main_category']

=== Model Cにのみ存在する特徴量 ===
['main_payment_type', 'max_payment_installments', 'payment_count', 'payment_type_count']


In [49]:
# STEP 52-10：CatBoost Model Eの4-Fold CV

from sklearn.metrics import average_precision_score, roc_auc_score

from src.model_training import train_classifier_model

cat_e_results = []

for fold_name, start, end in fold_configs:
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)

    train = df_cv_model.loc[
        (df_cv_model["delivered_timestamp"] < start)
        & (df_cv_model["last_review_timestamp"] < start)
    ]

    val = df_cv_model.loc[
        (df_cv_model["delivered_timestamp"] >= start)
        & (df_cv_model["delivered_timestamp"] < end)
    ]

    model, y_val, probabilities = train_classifier_model(
        train,
        val,
        features_e["numeric"],
        features_e["categorical"],
        classifiers["CatBoost"],
    )

    cat_e_results.append(
        {
            "model": "CatBoost",
            "features": "E",
            "fold": fold_name,
            "roc_auc": roc_auc_score(y_val, probabilities),
            "average_precision": average_precision_score(y_val, probabilities),
        }
    )

df_cat_e_results = pd.DataFrame(cat_e_results)

print("=== CatBoost Model E：Fold別性能 ===")
display(df_cat_e_results.round(6))


=== CatBoost Model E：Fold別性能 ===


,model,features,fold,roc_auc,average_precision
0,CatBoost,E,Fold 1,0.579281,0.128660
1,CatBoost,E,Fold 2,0.592308,0.119089
2,CatBoost,E,Fold 3,0.642660,0.171598
3,CatBoost,E,Fold 4,0.661137,0.217586


In [50]:
# 修正済みModel Dを含む5特徴量セットの比較

df_cat_feature_comparison = pd.concat(
    [
        df_cat_abc,
        df_cat_d_corrected.assign(
            model="CatBoost",
            features="D_corrected",
        ),
        df_cat_e_results,
    ],
    ignore_index=True,
)

print("=== CatBoost：特徴量セット別の平均性能 ===")
display(
    df_cat_feature_comparison.groupby("features")[["roc_auc", "average_precision"]]
    .agg(["mean", "std"])
    .round(6)
    .sort_values(("average_precision", "mean"), ascending=False)
)

print("\n=== Fold別ROC-AUC ===")
display(
    df_cat_feature_comparison.pivot(
        index="fold",
        columns="features",
        values="roc_auc",
    ).round(4)
)

print("\n=== Fold別AP ===")
display(
    df_cat_feature_comparison.pivot(
        index="fold",
        columns="features",
        values="average_precision",
    ).round(4)
)


=== CatBoost：特徴量セット別の平均性能 ===


roc_auc           average_precision          
                 mean       std              mean       std
features                                                   
C            0.620601  0.038015          0.161399  0.047055
D_corrected  0.621645  0.039619          0.160998  0.047545
E            0.618846  0.039265          0.159233  0.045108
B            0.604688  0.040087          0.151422  0.046767
A            0.601579  0.039761          0.148291  0.047526


=== Fold別ROC-AUC ===


features,A,B,C,D_corrected,E
fold,,,,,
Fold 1,0.5692,0.5687,0.5784,0.5831,0.5793
Fold 2,0.5682,0.5745,0.6001,0.5929,0.5923
Fold 3,0.6199,0.6232,0.6428,0.6474,0.6427
Fold 4,0.6490,0.6524,0.6611,0.6633,0.6611



=== Fold別AP ===


features,A,B,C,D_corrected,E
fold,,,,,
Fold 1,0.1285,0.1298,0.1294,0.1275,0.1287
Fold 2,0.1078,0.1111,0.1191,0.1183,0.1191
Fold 3,0.1402,0.1468,0.1755,0.1775,0.1716
Fold 4,0.2167,0.2181,0.2216,0.2207,0.2176


この結果を確認すると、商品カテゴリーの追加（E）は効果が高い一方、支払情報（B）や顧客の履歴（D）はそこまで効果が高くないことが分かった。Eを追加してみたが、やはりCが一番安定しているので、特徴量としてCを採用することとする。